In [0]:
import pandas as pd
import numpy as np

import mlflow
import os

from delta.tables import *

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
import seaborn as sns

## Load Data

In [0]:
# Load input data into a pandas DataFrame.
df_loaded =  spark.read.format('delta').load('dbfs:/tmp/new_tx_enhanced').toPandas()

# Preview data
df_loaded.head(5)

### Select supported columns
Select only the columns that are supported. This allows us to train a model that can predict on a dataset that has extra columns that are not used in training.
`["u_alias", "days_prior_to_last_transaction"]` are dropped in the pipelines. See the Alerts tab of the AutoML Experiment page for details on why these columns are dropped.

In [0]:
from databricks.automl_runtime.sklearn.column_selector import ColumnSelector
supported_cols = ["u_gender_group","u_ethnicity", "u_state",  "u_yearly_household_income", "u_age_group", "u_zipcode"]

## Preprocessors

### Numerical columns

Missing values for numerical columns are imputed with mean by default.

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

num_imputers = []
num_imputers.append(("impute_mean", SimpleImputer(), ["days_since_prior_transaction", "days_prior_to_last_transaction"]))

numerical_pipeline = Pipeline(steps=[
    ("converter", FunctionTransformer(lambda df: df.apply(pd.to_numeric, errors='coerce'))),
    ("imputers", ColumnTransformer(num_imputers)),
    ("standardizer", StandardScaler()),
])

numerical_transformers = [("numerical", numerical_pipeline, ["days_since_prior_transaction", "days_prior_to_last_transaction"])]

### Categorical columns

#### Low-cardinality categoricals
Convert each low-cardinality categorical column into multiple binary columns through one-hot encoding.
For each input categorical column (string or numeric), the number of output columns is equal to the number of unique values in the input column.

In [0]:
from databricks.automl_runtime.sklearn import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

one_hot_imputers = []

one_hot_pipeline = Pipeline(steps=[
    ("imputers", ColumnTransformer(one_hot_imputers, remainder="passthrough")),
    ("one_hot_encoder", OneHotEncoder(handle_unknown="indicator")),
])

categorical_one_hot_transformers = [("onehot", one_hot_pipeline, ["u_age_group", "u_ethnicity", "u_state", "u_yearly_household_income", "u_zipcode"])]

In [0]:
from sklearn.compose import ColumnTransformer

transformers = numerical_transformers + categorical_one_hot_transformers

preprocessor = ColumnTransformer(transformers, remainder="drop", sparse_threshold=1)

In [0]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("SVD", TruncatedSVD(n_components=10)),   
    ("cluster", KMeans(n_clusters=5, max_iter=1000))
])

# Fit the model to the loaded DataFrame
pipeline.fit(df_loaded)

# Predict clusters (optional)
cluster_labels = pipeline.predict(df_loaded)

# # If using MLflow, start an experiment and log parameters, metrics, or models as needed
# useremail = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
# experiment_name = f"/Users/{useremail}/segmentation"
# mlflow.set_experiment(experiment_name)
# with mlflow.start_run():
#     mlflow.log_param("n_clusters", 5)
#     # Log other parameters or metrics as needed
#     mlflow.sklearn.log_model(pipeline, "kmeans_model")

In [0]:
cluster_labels = pipeline.predict(df_loaded)
df_labeled = (
  pd.concat( 
    [df_loaded, pd.DataFrame(cluster_labels,columns=['cluster'])],
    axis=1
    )
  )
display(df_labeled)

In [0]:
df_labeled.groupby('cluster').count()

In [0]:
spark_df = spark.createDataFrame(df_labeled)
spark_df.write.format('delta').mode('overwrite').save('dbfs:/tmp/df_labeled')

### Iterate over Potential Values of K

In [0]:
pipeline_preprocessing = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("SVD", TruncatedSVD(n_components=10))
])

# Fit and transform the data
X_transformed = pipeline_preprocessing.fit_transform(df_loaded)  # Assuming df_loaded is your dataset

In [0]:
# broadcast features so that workers can access efficiently
X_broadcast = sc.broadcast(X_transformed)

# function to train model and return metrics
def evaluate_model(n):
  model = KMeans( n_clusters=n, init='k-means++', n_init=1, max_iter=10000)
  clusters = model.fit(X_broadcast.value).labels_
  return n, float(model.inertia_), float(silhouette_score(X_broadcast.value, clusters))

In [0]:
# define number of iterations for each value of k being considered
iterations = (
  spark
    .range(100) # iterations per value of k
    .crossJoin( spark.range(2,21).withColumnRenamed('id','n')) # cluster counts
    .repartition(sc.defaultParallelism)
    .select('n')
    .rdd
    )

In [0]:
# train and evaluate model for each iteration
results_pd = (
  spark
    .createDataFrame(
      iterations.map(lambda n: evaluate_model(n[0])), # iterate over each value of n
      schema=['n', 'inertia', 'silhouette']
      ).toPandas()
    )

# remove broadcast set from workers
X_broadcast.unpersist()

display(results_pd)

Databricks visualization. Run in Databricks to view.

Plotting inertia relative to n, *i.e.* the target number of clusters, we can see that the total sum of squared distances between cluster members and cluster centers decreases as we increase the number of clusters in our solution.  Our goal is not to drive inertia to zero (which would be achieved if we made each member the center of its own, 1-member cluster) but instead to identify the point in the curve where the incremental drop in inertia is diminished.  In our plot, we might identify this point as occurring somewhere around 10:

In [0]:
display(results_pd)

Databricks visualization. Run in Databricks to view.

In [0]:
total_iterations = 50000
n_for_bestofk = 10
X_broadcast = sc.broadcast(X_transformed)

def find_bestofk_for_partition(partition):
   
  # count iterations in this partition
  n_init = sum(1 for i in partition)
  
  # perform iterations to get best of k
  model = KMeans( n_clusters=n_for_bestofk, n_init=n_init, init='k-means++', max_iter=10000)
  model.fit(X_broadcast.value)
  
  # score model
  score = float(silhouette_score(X_broadcast.value, model.labels_))
  
  # return (score, model)
  yield (score, model)


# build RDD for distributed iteration
iterations = sc.range(
              total_iterations, 
              numSlices= sc.defaultParallelism * 4
              ) # distribute work into fairly even number of partitions that allow us to track progress
                        
# retrieve best of distributed iterations
bestofk_results = (
  iterations
    .mapPartitions(find_bestofk_for_partition)
    .sortByKey(ascending=False)
    .take(1)
    )[0]

# get score and model
bestofk_score = bestofk_results[0]
bestofk_model = bestofk_results[1]
bestofk_clusters = bestofk_model.labels_

In [0]:
# print best score obtained
print('Silhouette Score: {0:.6f}'.format(bestofk_score))

X_transformed_df = pd.DataFrame(X_transformed)

# combine households with cluster assignments
bestofk_labeled_X_pd = (
  pd.concat( 
    [X_transformed_df, pd.DataFrame(bestofk_clusters,columns=['cluster'])],
    axis=1
    )
  )
                        
# clean up 
X_broadcast.unpersist()

In [0]:
display(bestofk_labeled_X_pd )

In [0]:
bestofk_labeled_X_pd.columns = ['Dim_1', 'Dim_2'] + bestofk_labeled_X_pd.columns.tolist()[2:]

# Assuming 'n_for_bestofk' is the number of clusters in your best KMeans model
n_for_bestofk = bestofk_labeled_X_pd['cluster'].nunique()

# Visualize cluster assignments
fig, ax = plt.subplots(figsize=(10,8))
sns.scatterplot(
  data=bestofk_labeled_X_pd,
  x='Dim_1',
  y='Dim_2',
  hue='cluster',
  palette=[cm.nipy_spectral(float(i) / n_for_bestofk) for i in range(n_for_bestofk)],  # align colors with those used in silhouette plots
  legend='brief',
  alpha=0.5,
  ax=ax
)
_ = ax.legend(loc='lower right', ncol=1, fancybox=True)
plt.show()

In [0]:
from sklearn.metrics import silhouette_score, silhouette_samples

def plot_silhouette_chart(features, labels):
  
  n = len(np.unique(labels))
  
  # configure plot area
  fig, ax = plt.subplots(1, 1)
  fig.set_size_inches(8, 5)

  # configure plots for silhouette scores between -1 and 1
  ax.set_xlim([-0.1, 1])
  ax.set_ylim([0, len(features) + (n + 1) * 10])
  
  # avg silhouette score
  score = silhouette_score(features, labels)

  # compute the silhouette scores for each sample
  sample_silhouette_values = silhouette_samples(features, labels)

  y_lower = 10

  for i in range(n):

      # get and sort members by cluster and score
      ith_cluster_silhouette_values = sample_silhouette_values[labels == i]
      ith_cluster_silhouette_values.sort()

      # size y based on sample count
      size_cluster_i = ith_cluster_silhouette_values.shape[0]
      y_upper = y_lower + size_cluster_i

      # pretty up the charts
      color = cm.nipy_spectral(float(i) / n)
      
      ax.fill_betweenx(np.arange(y_lower, y_upper),
                        0, ith_cluster_silhouette_values,
                        facecolor=color, edgecolor=color, alpha=0.7)

      # label the silhouette plots with their cluster numbers at the middle
      ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

      # compute the new y_lower for next plot
      y_lower = y_upper + 10  # 10 for the 0 samples


  ax.set_title("Average silhouette of {0:.3f} with {1} clusters".format(score, n))
  ax.set_xlabel("The silhouette coefficient values")
  ax.set_ylabel("Cluster label")

  # vertical line for average silhouette score of all the values
  ax.axvline(x=score, color="red", linestyle="--")

  ax.set_yticks([])  # clear the yaxis labels / ticks
  ax.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
  
  return fig, ax

_ = plot_silhouette_chart(X_transformed_df, bestofk_clusters)